# Projet 4 — Détection de Fraude et de Triche en Jeu

**Master IADAIG — ENS Fès**  
**Objectif :** construire un pipeline de Machine Learning capable d'identifier automatiquement les joueurs suspects dans un environnement de jeu en ligne.

Ce notebook est organisé étape par étape :

1. Chargement du dataset  
2. Nettoyage des données  
3. Feature engineering  
4. Exploration des données  
5. Prétraitement : encoding + scaling  
6. Modèles supervisés  
7. Modèles non supervisés  
8. Autoencoder  
9. Comparaison finale  
10. Interprétation

## 1. Importation des bibliothèques

Nous utilisons principalement :

- `pandas` pour manipuler les données.
- `numpy` pour les calculs numériques.
- `matplotlib` pour les visualisations.
- `scikit-learn` pour les modèles de Machine Learning.
- `joblib` pour sauvegarder les modèles entraînés.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

RANDOM_STATE = 42

## 2. Configuration des chemins

Le notebook est reproductible : il lit le dataset depuis le dossier `data/` et sauvegarde automatiquement les figures, résultats et modèles.

In [ ]:
# Le notebook se trouve dans notebooks/, donc le dossier racine du projet est le parent.
PROJECT_DIR = Path.cwd()

# Si on exécute le notebook depuis notebooks/, on remonte d'un niveau.
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

# Dataset placé dans le même dossier que ce notebook
DATA_PATH = PROJECT_DIR / "online_gaming_behavior.csv"

# Dossiers créés automatiquement après exécution
FIGURES_DIR = PROJECT_DIR / "figures"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
MODELS_DIR = PROJECT_DIR / "models"

for folder in [FIGURES_DIR, OUTPUTS_DIR, MODELS_DIR]:
    folder.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset path:", DATA_PATH)

## 3. Chargement du dataset

Le dataset contient des informations comportementales sur des joueurs : durée de session, APM, KD ratio, précision, taux de headshots, mode de jeu, région, rang, items et probabilité de triche.

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

## 4. Compréhension des variables

Cette étape permet de voir :

- Le type de chaque colonne.
- Le nombre de valeurs non nulles.
- Les variables numériques et catégorielles.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["Cheat_Likelihood"].value_counts()

### Interprétation

La variable cible est `Cheat_Likelihood` :

- `0` : joueur normal.
- `1` : joueur suspect.

Le dataset est déséquilibré : les joueurs suspects sont minoritaires.  
Donc l'accuracy seule ne suffit pas. Il faut aussi utiliser la precision, le recall, le F1-score et la matrice de confusion.

## 5. Nettoyage des données

La colonne `Items` contient des valeurs manquantes.  
Au lieu de supprimer les lignes, nous remplaçons les valeurs manquantes par `No_Item`.

Cette solution garde toutes les observations et indique qu'aucun item n'a été enregistré.

In [ ]:
# Valeurs manquantes avant nettoyage
df.isna().sum()

In [ ]:
# Nettoyage de la colonne Items
df["Items"] = df["Items"].fillna("No_Item")

# Vérification après nettoyage
df.isna().sum()

## 6. Feature Engineering

Nous ajoutons trois nouvelles variables pour mieux représenter le comportement du joueur :

- `Aim_Score = Accuracy × Headshot_Rate`
- `Combat_Score = KD_Ratio × Accuracy`
- `Activity_Score = APM / Session_Duration`

Ces variables combinent plusieurs signaux utiles pour détecter les comportements suspects.

In [ ]:
df["Aim_Score"] = df["Accuracy"] * df["Headshot_Rate"]
df["Combat_Score"] = df["KD_Ratio"] * df["Accuracy"]
df["Activity_Score"] = df["APM"] / df["Session_Duration"]

df.head()

In [ ]:
# Sauvegarder le dataset nettoyé et enrichi
cleaned_path = OUTPUTS_DIR / "online_gaming_behavior_cleaned_features.csv"
df.to_csv(cleaned_path, index=False)
print("Dataset sauvegardé:", cleaned_path)

## 7. EDA — Exploration des données

L'EDA permet de comprendre la structure du dataset et d'identifier les variables qui séparent les joueurs normaux des joueurs suspects.

In [ ]:
plt.figure(figsize=(6, 4))
df["Cheat_Likelihood"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribution des classes")
plt.xlabel("Cheat_Likelihood (0 = normal, 1 = suspect)")
plt.ylabel("Nombre de joueurs")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_distribution_classes.png", dpi=160)
plt.show()

### Interprétation

La classe `0` est très majoritaire par rapport à la classe `1`.  
Cela confirme que le dataset est déséquilibré.  
Dans un système anti-cheat, le **recall** est très important car on veut détecter le maximum de tricheurs.

In [ ]:
numeric_cols = [
    "Session_Duration", "APM", "KD_Ratio", "Accuracy",
    "Headshot_Rate", "Aim_Score", "Combat_Score", "Activity_Score"
]

plt.figure(figsize=(11, 5))
df[numeric_cols].boxplot()
plt.title("Boxplots des variables numériques")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_boxplots_variables_numeriques.png", dpi=160)
plt.show()

In [ ]:
corr = df[numeric_cols + ["Cheat_Likelihood"]].corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr)
plt.title("Matrice de corrélation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar()

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_matrice_correlation.png", dpi=160)
plt.show()

### Interprétation de la corrélation

Les variables `KD_Ratio`, `Headshot_Rate`, `Accuracy`, `Aim_Score` et `Combat_Score` sont très liées à la variable cible.  
Cela est logique : un joueur suspect peut avoir une précision, un taux de headshots ou un KD ratio anormalement élevés.

In [ ]:
for col in ["APM", "Accuracy", "Headshot_Rate", "KD_Ratio", "Aim_Score", "Combat_Score"]:
    plt.figure(figsize=(7, 4))
    plt.hist(df[df["Cheat_Likelihood"] == 0][col], bins=30, alpha=0.7, label="Normal")
    plt.hist(df[df["Cheat_Likelihood"] == 1][col], bins=30, alpha=0.7, label="Suspect")
    plt.title(f"Distribution de {col} selon la classe")
    plt.xlabel(col)
    plt.ylabel("Fréquence")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"hist_{col}.png", dpi=160)
    plt.show()

## 8. Prétraitement : Encoding + Scaling

Avant l'entraînement :

- On supprime `Player_ID` et `Game_Session_ID` car ce sont des identifiants.
- On sépare `X` et `y`.
- On applique `StandardScaler` aux variables numériques.
- On applique `OneHotEncoder` aux variables catégorielles.
- On fait un train/test split stratifié pour garder la même proportion de joueurs suspects.

In [ ]:
X = df.drop(columns=["Cheat_Likelihood", "Player_ID", "Game_Session_ID"])
y = df["Cheat_Likelihood"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Variables numériques:", numeric_features)
print("Variables catégorielles:", categorical_features)

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Fraud ratio train:", y_train.mean())
print("Distribution train:")
print(y_train.value_counts())
print("Distribution test:")
print(y_test.value_counts())

## 9. Fonctions d'évaluation

Ces fonctions permettent de calculer les métriques et d'afficher les matrices de confusion d'une manière propre.

In [ ]:
def get_metrics(name, y_true, y_pred, y_score=None, model_type=""):
    return {
        "Model": name,
        "Type": model_type,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_score) if y_score is not None else np.nan
    }

def plot_confusion_matrix(y_true, y_pred, title, filename=None):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Prédiction")
    plt.ylabel("Valeur réelle")
    plt.xticks([0, 1], ["Normal", "Suspect"])
    plt.yticks([0, 1], ["Normal", "Suspect"])
    plt.colorbar()

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()

    if filename is not None:
        plt.savefig(FIGURES_DIR / filename, dpi=160)

    plt.show()

def make_dense(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else matrix

## 10. Modèle 1 — Logistic Regression

La Logistic Regression est utilisée comme modèle de base.  
Nous utilisons `class_weight="balanced"` pour tenir compte du déséquilibre entre les classes.

In [ ]:
log_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_score_log = log_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test, y_score_log))

plot_confusion_matrix(
    y_test,
    y_pred_log,
    "Logistic Regression",
    "confusion_logistic_regression.png"
)

joblib.dump(log_model, MODELS_DIR / "logistic_regression.joblib")

## 11. Modèle 2 — Random Forest

Random Forest est un modèle supervisé puissant, basé sur plusieurs arbres de décision.  
Il est adapté aux relations non linéaires entre les variables.

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_score_rf = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_score_rf))

plot_confusion_matrix(
    y_test,
    y_pred_rf,
    "Random Forest",
    "confusion_random_forest.png"
)

joblib.dump(rf_model, MODELS_DIR / "random_forest.joblib")

## 12. Préparation des modèles non supervisés

Pour `Isolation Forest`, `One-Class SVM` et `Autoencoder`, on entraîne les modèles uniquement sur les joueurs normaux.  
L'objectif est d'apprendre le comportement normal, puis de considérer comme suspect ce qui s'éloigne de ce comportement.

In [ ]:
X_train_normal = X_train[y_train == 0]

preprocess_unsup = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_train_normal_scaled = preprocess_unsup.fit_transform(X_train_normal)
X_test_scaled = preprocess_unsup.transform(X_test)

fraud_ratio = y_train.mean()

print("Nombre de joueurs normaux dans train:", X_train_normal.shape[0])
print("Fraud ratio utilisé comme contamination/nu:", fraud_ratio)

## 13. Modèle 3 — Isolation Forest

Isolation Forest isole les observations rares ou différentes.  
Dans ce projet, les joueurs suspects sont considérés comme des anomalies.

In [ ]:
iso_model = IsolationForest(
    n_estimators=300,
    contamination=fraud_ratio,
    random_state=RANDOM_STATE
)

iso_model.fit(X_train_normal_scaled)

raw_pred_iso = iso_model.predict(X_test_scaled)
y_pred_iso = np.where(raw_pred_iso == -1, 1, 0)

print(classification_report(y_test, y_pred_iso))

plot_confusion_matrix(
    y_test,
    y_pred_iso,
    "Isolation Forest",
    "confusion_isolation_forest.png"
)

joblib.dump(iso_model, MODELS_DIR / "isolation_forest.joblib")

## 14. Modèle 4 — One-Class SVM

One-Class SVM apprend une frontière autour du comportement normal.  
Les observations en dehors de cette frontière sont considérées comme suspectes.

In [ ]:
ocsvm_model = OneClassSVM(
    nu=fraud_ratio,
    kernel="rbf",
    gamma="scale"
)

ocsvm_model.fit(X_train_normal_scaled)

raw_pred_svm = ocsvm_model.predict(X_test_scaled)
y_pred_svm = np.where(raw_pred_svm == -1, 1, 0)

print(classification_report(y_test, y_pred_svm))

plot_confusion_matrix(
    y_test,
    y_pred_svm,
    "One-Class SVM",
    "confusion_oneclass_svm.png"
)

joblib.dump(ocsvm_model, MODELS_DIR / "one_class_svm.joblib")
joblib.dump(preprocess_unsup, MODELS_DIR / "preprocess_unsupervised.joblib")

## 15. Modèle 5 — Autoencoder

Un Autoencoder apprend à reconstruire les données normales.  
Si un joueur a un comportement anormal, l'erreur de reconstruction devient plus élevée.

Ici, on utilise un Autoencoder simple avec `MLPRegressor` pour éviter d'ajouter des bibliothèques lourdes comme TensorFlow.

In [ ]:
X_train_normal_dense = make_dense(X_train_normal_scaled)
X_test_dense = make_dense(X_test_scaled)

autoencoder = MLPRegressor(
    hidden_layer_sizes=(16, 8, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=RANDOM_STATE
)

# Entraînement : entrée = sortie, uniquement sur joueurs normaux
autoencoder.fit(X_train_normal_dense, X_train_normal_dense)

# Reconstruction
train_reconstruction = autoencoder.predict(X_train_normal_dense)
test_reconstruction = autoencoder.predict(X_test_dense)

# Erreurs de reconstruction
train_errors = np.mean((X_train_normal_dense - train_reconstruction) ** 2, axis=1)
test_errors = np.mean((X_test_dense - test_reconstruction) ** 2, axis=1)

# Seuil basé sur le taux de fraude observé dans le train
threshold = np.quantile(train_errors, 1 - fraud_ratio)

# Si erreur > seuil => suspect
y_pred_auto = (test_errors > threshold).astype(int)

print("Seuil de reconstruction:", threshold)
print(classification_report(y_test, y_pred_auto))

plot_confusion_matrix(
    y_test,
    y_pred_auto,
    "Autoencoder",
    "confusion_autoencoder.png"
)

joblib.dump(autoencoder, MODELS_DIR / "autoencoder_mlp.joblib")

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(test_errors[y_test.values == 0], bins=30, alpha=0.7, label="Normal")
plt.hist(test_errors[y_test.values == 1], bins=30, alpha=0.7, label="Suspect")
plt.axvline(threshold, linestyle="--", label="Seuil")
plt.title("Autoencoder — erreurs de reconstruction")
plt.xlabel("Erreur de reconstruction")
plt.ylabel("Fréquence")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "autoencoder_reconstruction_errors.png", dpi=160)
plt.show()

## 16. Comparaison finale des modèles

Nous comparons les modèles avec plusieurs métriques.  
Dans un système anti-cheat, le recall est très important, car un faux négatif signifie qu'un tricheur n'a pas été détecté.

In [ ]:
results = pd.DataFrame([
    get_metrics("Logistic Regression", y_test, y_pred_log, y_score_log, "Supervised"),
    get_metrics("Random Forest", y_test, y_pred_rf, y_score_rf, "Supervised"),
    get_metrics("Isolation Forest", y_test, y_pred_iso, None, "Unsupervised"),
    get_metrics("One-Class SVM", y_test, y_pred_svm, None, "Unsupervised"),
    get_metrics("Autoencoder", y_test, y_pred_auto, None, "Unsupervised")
])

results = results.sort_values("F1-score", ascending=False)
results

In [ ]:
results.to_csv(OUTPUTS_DIR / "model_comparison_results.csv", index=False)
print("Résultats sauvegardés dans:", OUTPUTS_DIR / "model_comparison_results.csv")

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(results["Model"], results["F1-score"])
plt.title("Comparaison des modèles selon F1-score")
plt.ylabel("F1-score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "comparison_f1_score.png", dpi=160)
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(results["Model"], results["Recall"])
plt.title("Comparaison des modèles selon Recall")
plt.ylabel("Recall")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "comparison_recall.png", dpi=160)
plt.show()

## 17. Courbes ROC pour les modèles supervisés

Les courbes ROC sont utilisées pour les modèles qui produisent des probabilités.

In [ ]:
plt.figure(figsize=(7, 5))

for name, score in [
    ("Logistic Regression", y_score_log),
    ("Random Forest", y_score_rf)
]:
    fpr, tpr, _ = roc_curve(y_test, score)
    plt.plot(fpr, tpr, label=name)

plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.title("Courbes ROC — modèles supervisés")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "roc_curves_supervised.png", dpi=160)
plt.show()

## 18. Importance des variables — Random Forest

Cette étape permet de savoir quelles variables influencent le plus la décision du modèle Random Forest.

In [ ]:
feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()
importances = rf_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

importance_df.to_csv(OUTPUTS_DIR / "random_forest_feature_importance.csv", index=False)

importance_df.head(15)

In [ ]:
top_importance = importance_df.head(15)

plt.figure(figsize=(9, 5))
plt.barh(top_importance["Feature"][::-1], top_importance["Importance"][::-1])
plt.title("Top 15 variables importantes — Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "random_forest_feature_importance.png", dpi=160)
plt.show()

## 19. Interprétation générale

Les modèles supervisés profitent directement des labels `Cheat_Likelihood`, ce qui explique leurs performances très élevées.  
Ces résultats doivent être interprétés avec prudence, car le dataset présente une séparation claire entre joueurs normaux et joueurs suspects.

Les modèles non supervisés restent importants dans un contexte réel, car les labels ne sont pas toujours disponibles.  
`One-Class SVM` et `Autoencoder` sont intéressants si l'objectif est de détecter le maximum de joueurs suspects, même au prix de faux positifs.  
`Isolation Forest` offre une autre approche de détection d'anomalies basée sur l'isolement des observations rares.

## 20. Conclusion

Ce notebook présente un pipeline complet de détection de fraude et de triche en jeu :

- Nettoyage du dataset.
- Feature engineering.
- EDA.
- Prétraitement avec encoding et scaling.
- Modélisation supervisée.
- Détection d'anomalies.
- Autoencoder.
- Comparaison et interprétation des modèles.

Dans un projet anti-cheat réel, il est recommandé d'utiliser une approche hybride : modèles supervisés lorsque les labels sont disponibles, et modèles non supervisés comme système d'alerte lorsque les labels sont absents.